# 01 — Explore contrasts

Load a raw PS-OCT section, a strip of registered sections from each brain and the aligned 3D volume; check variables, shapes, types and ranges; and view the four contrasts side by side to decide which channels to use. Data: `Derivatives/<subject>/PS-OCT/Enface/` in the brainlife project (Moe: coronal sections; Vlad: sagittal sections), stored under `data/moe` and `data/vlad`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

from scipy.io import loadmat

from brainseg.preprocess import normalize

DATA = Path("../data")
CONTRASTS = {"Reflectivity": "Ref", "Cross-pol": "Cr", "Retardance": "R", "Orientation": "AO"}


def load_mat(path):
    return {k: v for k, v in loadmat(path).items() if not k.startswith("__")}


def load_nifti(path):
    return np.asarray(nib.load(path).dataobj)


def describe(name, a):
    print(f"{name:24s} {str(a.shape):16s} {str(a.dtype):8s} "
          f"[{np.nanmin(a):8.4g}, {np.nanmax(a):8.4g}]  NaN {np.isnan(a).mean():6.1%}")


def draw(ax, name, img):
    if name == "Orientation":  # angle: cyclic colormap, no percentile clipping
        return ax.imshow(img, cmap="twilight", vmin=np.nanmin(img), vmax=np.nanmax(img))
    return ax.imshow(normalize(img), cmap="gray")


def show(images, title):
    fig, axes = plt.subplots(1, len(images), figsize=(4.5 * len(images), 4), layout="constrained")
    for ax, (name, img) in zip(axes, images.items()):
        fig.colorbar(draw(ax, name, img), ax=ax, shrink=0.7)
        ax.set(title=name, xticks=[], yticks=[])
    fig.suptitle(title)

## Raw section

Full-resolution en-face section (`Enface/<contrast>/Slice_160_En*.mat`), Moe. Orientation is complex-valued (phase = angle). Cast to float32 on load; shown at 1/4 resolution.

In [ ]:
raw = {}
for name, tag in CONTRASTS.items():
    for var, a in load_mat(DATA / "moe" / "mat" / f"Slice_160_En{tag}.mat").items():
        if np.iscomplexobj(a):  # orientation is stored as a complex number; its phase is the angle
            describe(f"{name}: |{var}|", np.abs(a))
            a = np.angle(a)
        raw[name] = a.astype(np.float32)
        describe(f"{name}: {var}", raw[name])

show({k: v[::4, ::4] for k, v in raw.items()}, "Moe slice 160, raw")

In [ ]:
# Full-resolution crop to see fine structure (cortex, vessels, seams)
r, c = raw["Retardance"].shape[0] // 3, raw["Retardance"].shape[1] // 3
show({k: v[r:r + 1500, c:c + 1500] for k, v in raw.items()}, "Moe slice 160, 1500 px crop at full resolution")

## Sections across each brain

Registered sections, 10× downsampled (`Registration/<contrast>/lowres/`), evenly spaced front to back, to see how contrast and artifacts vary between sections.

In [ ]:
def load_lowres(subject):
    """Return {slice number: {contrast: image}} for the downloaded low-res sections."""
    sections = {}
    for path in sorted((DATA / subject / "lowres").glob("Slice_*_EnRef_*.nii.gz")):
        n = int(path.name.split("_")[1])
        sections[n] = {name: np.squeeze(load_nifti(str(path).replace("EnRef", f"En{tag}"))).T
                       for name, tag in CONTRASTS.items()}
    return sections


def show_strip(subject, sections):
    fig, axes = plt.subplots(len(CONTRASTS), len(sections), figsize=(2 * len(sections), 2 * len(CONTRASTS)),
                             layout="constrained")
    for col, (n, images) in enumerate(sections.items()):
        for row, (name, img) in enumerate(images.items()):
            draw(axes[row, col], name, img)
            axes[row, col].set(xticks=[], yticks=[])
            axes[0, col].set_title(f"slice {n}")
            axes[row, 0].set_ylabel(name)
    fig.suptitle(subject.capitalize())


for subject in ["moe", "vlad"]:
    sections = load_lowres(subject)
    n = next(iter(sections))
    for name, img in sections[n].items():
        describe(f"{subject} slice {n}: {name}", img)
    print("voxel size (mm):", nib.load(next((DATA / subject / "lowres").glob("*.nii.gz"))).header.get_zooms(), "\n")
    show_strip(subject, sections)

## 3D volume

Slide deck of all registered sections of Moe, resampled into its MRI space (0.4 mm).

In [ ]:
VOLUME = DATA / "moe" / "volume_in_MRI" / "{}_slide_deck_in_MRI.nii.gz"
VOL_TAGS = {"Reflectivity": "Ref", "Cross-pol": "Cro", "Retardance": "Ret", "Orientation": "Ori"}

volume = {}
for name, tag in VOL_TAGS.items():
    img = nib.load(str(VOLUME).format(tag))
    volume[name], affine = np.asarray(img.dataobj), img.affine
    describe(name, volume[name])

print("orientation:", nib.aff2axcodes(affine), " voxel size (mm):", nib.affines.voxel_sizes(affine))

for axis, plane in enumerate(nib.aff2axcodes(affine)):
    mid = volume["Reflectivity"].shape[axis] // 2
    show({k: np.rot90(np.take(v, mid, axis)) for k, v in volume.items()}, f"axis {axis} ({plane}) = {mid}")